## Visual-LLM Prediction generation

#### Setup

In [ ]:
# 1) Imports & Config
import os, io, time, json, base64, re
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

import requests
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
Image.MAX_IMAGE_PIXELS = None              # disable decompression bomb limit (trusted images only)
ImageFile.LOAD_TRUNCATED_IMAGES = True     # tolerate slightly corrupted files
from tqdm import tqdm
import matplotlib.pyplot as plt
import csv


# --- OpenRouter config ---
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")
BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
DEFAULT_HEADERS = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "HTTP-Referer": "https://example.com",  # optional; replace with your app/site
    "X-Title": "VegCover-Quadrat-Benchmark"
}


In [ ]:

def load_and_normalize_image(path: Path, max_side: int = 1536) -> Image.Image:
    img = Image.open(path)
    img = img.convert("RGB")
    # Downscale without ever expanding:
    img.thumbnail((max_side, max_side), Image.BICUBIC)
    return img


def pil_to_base64_jpeg(img: Image.Image, quality: int = 90) -> str:
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality)
    return base64.b64encode(buf.getvalue()).decode("utf-8")

#### Models and Prompts

In [ ]:
ENABLED_MODELS = [
    "google/gemma-3-12b-it:free",
    "google/gemma-3-27b-it:free",
    "meta-llama/llama-4-maverick:free",
    "mistralai/mistral-small-3.2-24b-instruct:free",
    "meta-llama/llama-4-scout:free",
    "qwen/qwen2.5-vl-32b-instruct:free",
    ]
print("Models:", ENABLED_MODELS)

In [ ]:
PROMPTS = {
    "v1_point_hint": (
        "Estimate % vegetation cover inside the 1x1 m quadrat only.\n"
        "Imagine sampling many points uniformly and counting vegetation hits.\n\n"
        "Return STRICT JSON only:\n"
        "{\"vegetation_percent\": <number>, \"confidence\": <number>}\n"
    ),
    "v2_short": (
        "Inside the quadrat only: estimate vegetation cover percent (0–100).\n"
        "Return JSON only:\n"
        "{\"vegetation_percent\": <number>, \"confidence\": <number>}\n"
    ),
    "v3_detailed_ecology": (
        "You are an expert in vegetation ecology, trained in standard field methods such as quadrat sampling and point-intercept analysis.\n"
        "Your task is to estimate the percentage of ground area covered by live vegetation inside a 1x1 m quadrat.\n\n"
        "Definition of vegetation cover (based on ecological literature):\n"
        "- Include: living grasses, herbs, shrubs, tree seedlings, mosses, and any other green photosynthetic plant tissue.\n"
        "- Exclude: bare soil, litter (dead leaves, twigs), rocks, shadows, water, and man-made objects.\n"
        "- Count overlapping vegetation only once (do not double-count leaves stacked vertically).\n"
        "- Boundaries: only consider the area strictly inside the quadrat frame; ignore anything outside.\n\n"
        "Provide your best estimate of the proportion of ground covered by vegetation (0–100%).\n"
        "Return STRICT JSON only:\n"
        "{\"vegetation_percent\": <number>, \"confidence\": <number>}\n"
    ),
    "v4_grid_overlay": (
        "Estimate % vegetation cover inside the 1x1 m quadrat only.\n"
        "One way to do this is to imagine dividing the quadrat into a 10x10 grid (100 equal squares).\n"
        "For each square, decide if vegetation covers most of it or not, then sum the total.\n"
        "This approximates the area covered by vegetation.\n\n"
        "Return STRICT JSON only:\n"
        "{\"vegetation_percent\": <number>, \"confidence\": <number>}\n"
    ),
}

##### Path - Getting Images and Reference Values

In [ ]:
# Paths
IMAGES_DIR = Path("../data/pics/All")  # INPUT DIRECTORY
OUTPUTS_DIR = Path("../outputs"); OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
GROUND_TRUTH_CSV = Path("../data/refs/All.csv")

# Controls
DRY_RUN = False            # set to False to make real API calls
GEN_PARAMS = dict(temperature=0, top_p=1)
MAX_IMAGES = None          # set to an int to subset
TIMEOUT_S = 90
RETRIES_ON_JSON_FAIL = 1

ref = pd.read_csv(GROUND_TRUTH_CSV)
ref_filtered = ref.loc[ref['Veg %'].notna(), ['Filename', 'Veg %']].copy()

def list_images(img_dir: Path, allowed_names=None) -> list[Path]:
    """
    List images in a directory, filtered by allowed_names if provided.
    allowed_names should be a set of filenames (strings).
    """
    exts = {".jpg", ".jpeg", ".png", ".webp"}
    files = [p for p in img_dir.iterdir() if p.suffix.lower() in exts]

    if allowed_names is not None:
        files = [p for p in files if p.name in allowed_names]

    files.sort()
    return files[:MAX_IMAGES] if MAX_IMAGES else files

# Build the set of filenames with reference values
allowed_filenames = set(ref_filtered["Filename"].astype(str).tolist())

# Get only those images that appear in ref_filtered
image_paths = list_images(IMAGES_DIR, allowed_names=allowed_filenames)

print(f"Found {len(image_paths)} images in {IMAGES_DIR.resolve()} (filtered by ref list)")
print("First few:", [p.name for p in image_paths[:10]])


In [ ]:
ref_filtered[ref_filtered["Filename"]=='IMG_20250421_125020.jpg']

##### Requests to OpenRouter

### Parsing a response

Responses are reduced to a cover percentage and a confidence value by the
parser published in `parser/fvc_parser.py`, the same one used for the local
pathways and for every published estimate.

It scans the response for well-formed JSON objects and takes the last one
carrying both required fields as numbers, which accommodates a model that
narrates its reasoning or restates the requested output format before
answering. Confidence is returned exactly as the model wrote it. The scale
differs between models, so normalisation onto a common interval is a separate
and recorded step further down.


In [ ]:
import sys, pathlib

# locate parser/ by walking up from the working directory, so the notebook runs
# from anywhere inside the repository
for _c in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_c / "parser" / "fvc_parser.py").is_file():
        sys.path.insert(0, str(_c / "parser"))
        break

# same (cover, confidence)-or-None contract the next cell already expects
from fvc_parser import parse_response as parse_strict_json

In [ ]:
def call_openrouter(model_id: str,
                    prompt: str,
                    img_b64: str,
                    allow_response_format: bool = True,
                    max_retries: int = 5,
                    base_delay: int = 5) -> dict:
    """
    Call OpenRouter with built-in exponential backoff on rate limits (429)
    and transient network/server errors.
    """
    t0 = time.time()

    # DRY RUN shortcut
    if DRY_RUN:
        vp = float(np.clip(np.random.normal(52, 18), 0, 100))
        cf = float(np.clip(np.random.uniform(0.55, 0.95), 0, 1))
        raw = json.dumps({"vegetation_percent": vp, "confidence": cf})
        return {
            "vegetation_percent": vp,
            "confidence": cf,
            "raw_text": raw,
            "usage": {},
            "latency_ms": int((time.time()-t0)*1000),
        }

    if not OPENROUTER_API_KEY:
        raise RuntimeError("OPENROUTER_API_KEY is not set. Export it before running.")

    headers = DEFAULT_HEADERS.copy()

    payload = {
        "model": model_id,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}}
            ]
        }],
        "temperature": GEN_PARAMS.get("temperature", 0),
        # "top_p": GEN_PARAMS.get("top_p", 1),   # keep commented if you don't need it
        # "max_tokens": 200,
    }

    url = f"{BASE_URL}/chat/completions"

    # --- retry loop for network/429 errors ---
    for attempt in range(max_retries):
        try:
            r = requests.post(url, headers=headers, json=payload, timeout=TIMEOUT_S)
            r.raise_for_status()   # raises for 4xx/5xx
            data = r.json()
            break   # success -> exit retry loop
        except requests.exceptions.HTTPError as e:
            code = getattr(e.response, "status_code", None)
            if code == 429 or (code and 500 <= code < 600):
                wait = base_delay * (2 ** attempt)
                print(f"[call_openrouter] {code} error -> retry {attempt+1}/{max_retries} after {wait}s")
                time.sleep(wait)
                continue
            raise   # other HTTP errors: stop immediately
        except requests.exceptions.RequestException as e:
            # transient network error
            if attempt < max_retries - 1:
                wait = base_delay * (2 ** attempt)
                print(f"[call_openrouter] network error -> retry {attempt+1}/{max_retries} after {wait}s")
                time.sleep(wait)
                continue
            raise
    else:
        raise RuntimeError(f"Failed after {max_retries} retries for model {model_id}")

    # --- parsing & post-processing ---
    text = data["choices"][0]["message"]["content"]
    usage = data.get("usage", {})
    parsed = parse_strict_json(text)

    # Retry once without response_format if parsing failed (if you ever reintroduce it)
    if (parsed is None) and allow_response_format:
        # We can optionally remove any response_format key if we add it back later
        return call_openrouter(model_id, prompt, img_b64, allow_response_format=False)

    if parsed is None:
        vp, cf = (np.nan, np.nan)
    else:
        vp, cf = parsed

    return {
        "vegetation_percent": vp,
        "confidence": cf,
        "raw_text": text,
        "usage": usage,
        "latency_ms": int((time.time() - t0) * 1000),
    }

##### Batch Runner

In [ ]:
# Build a normalized filename -> reference map from your ref_filtered DataFrame
def norm_name(s: str) -> str:
    return str(s).strip().lower()

ref_map = {
    norm_name(fn): ref
    for fn, ref in zip(ref_filtered["Filename"].astype(str), ref_filtered["Veg %"])
}

# (Optional) sanity check: warn if any selected image lacks a reference
missing = [p.name for p in image_paths if norm_name(p.name) not in ref_map]
if missing:
    print(f"Warning: {len(missing)} images have no reference in ref_filtered (first 10):", missing[:10])

In [ ]:
# Path to your external reference CSV (edit this!)
# REFERENCE_CSV = Path("path/to/your/reference.csv")

def norm_name(name: str) -> str:
    """
    Whatever normalization you already use.
    For example:
    return name.strip().lower()
    """
    return name.strip()  # <-- or your existing implementation

# --- Load external reference CSV ---
ref_df = pd.read_csv(GROUND_TRUTH_CSV)

# The first column in your example is just an index (374, 375, ...)
# so columns will likely be: ['Unnamed: 0','Campaign','Site','Date','Time','Filename','Veg %']

# Clean up column names a bit (optional but handy)
ref_df = ref_df.rename(columns=lambda c: c.strip())

# Drop rows with no Veg % (if you want to ignore missing labels)
ref_df = ref_df.dropna(subset=["Veg %"])

# Build a normalized filename column to match against img_path.name
ref_df["filename_norm"] = ref_df["Filename"].apply(norm_name)

# If there are duplicate filenames, keep the first occurrence
ref_df = ref_df.drop_duplicates(subset=["filename_norm"], keep="first")

# Finally, build the mapping: normalized filename -> Veg %
ref_map = dict(zip(ref_df["filename_norm"], ref_df["Veg %"]))

print(f"Loaded {len(ref_map)} reference entries from {GROUND_TRUTH_CSV}")


In [ ]:
ref_df[ref_df["Filename"]=='IMG_20250421_125020.jpg']


In [ ]:
print("Found", len(image_paths), "images:")
print([p.name for p in image_paths[:10]])

ENABLED_PROMPTS = list(PROMPTS.keys())

logs_csv = OUTPUTS_DIR / "logs_openrouter.csv"
logs_jsonl = OUTPUTS_DIR / "logs_openrouter.jsonl"

# --- Build a set of completed triples and keep previous successful rows ---
done = set()
previous_ok_df = pd.DataFrame()   # store past successes to merge later

if logs_csv.exists():
    try:
        prev = pd.read_csv(logs_csv)
        # keep only rows with no error recorded
        prev_ok = prev[prev["error"].isna() | (prev["error"] == "")]
        done = set(map(tuple, prev_ok[["image","prompt_id","model"]].itertuples(index=False, name=None)))
        previous_ok_df = prev_ok.copy()  # ✅ keep for later merge
        print(f"Resuming: {len(done)} successful rows already present in CSV.")
    except Exception as e:
        print("Could not read existing CSV for resume:", e)
else:
    prev_ok = pd.DataFrame(columns=["image","prompt_id","model"])

# --- Open files in append mode and create CSV header if needed ---
import csv, os, json
csv_exists = logs_csv.exists()
jf = open(logs_jsonl, "a", encoding="utf-8")
cf = open(logs_csv, "a", newline="", encoding="utf-8")
csv_writer = None

csv_fieldnames = [
    "image","prompt_id","model","vegetation_percent","confidence",
    "raw_text","latency_ms","error","reference"
]

if not csv_exists:
    csv_writer = csv.DictWriter(cf, fieldnames=csv_fieldnames)
    csv_writer.writeheader()
else:
    csv_writer = csv.DictWriter(cf, fieldnames=csv_fieldnames)

def _persist_row(row: dict):
    """Append one row to JSONL and CSV, flushing to disk each time."""
    jf.write(json.dumps(row, ensure_ascii=False) + "\n")
    jf.flush(); os.fsync(jf.fileno())
    csv_row = {k: row.get(k, None) for k in csv_fieldnames}
    csv_writer.writerow(csv_row)
    cf.flush(); os.fsync(cf.fileno())

rows_new = []  # keep only the new rows from this run

try:
    for img_path in tqdm(image_paths, desc="Images"):
        ref_value = ref_map.get(norm_name(img_path.name), None)
        pil_img = load_and_normalize_image(img_path)
        b64 = pil_to_base64_jpeg(pil_img)

        for prompt_id in ENABLED_PROMPTS:
            prompt = PROMPTS[prompt_id]
            for model_id in ENABLED_MODELS:
                triplet = (img_path.name, prompt_id, model_id)
                if triplet in done:
                    continue  # already processed (resume)

                # try:
                #     res = call_openrouter(model_id, prompt, b64)
                #     out = {
                #         "image": img_path.name,
                #         "prompt_id": prompt_id,
                #         "model": model_id,
                #         "vegetation_percent": res.get("vegetation_percent"),
                #         "confidence": res.get("confidence"),
                #         "raw_text": res.get("raw_text"),
                #         "latency_ms": res.get("latency_ms"),
                #         "error": None,
                #         "reference": ref_value,
                #     }
                #     done.add(triplet)
                # except Exception as e:
                #     out = {
                #         "image": img_path.name,
                #         "prompt_id": prompt_id,
                #         "model": model_id,
                #         "vegetation_percent": None,
                #         "confidence": None,
                #         "raw_text": None,
                #         "latency_ms": None,
                #         "error": str(e),
                #         "reference": ref_value,
                #     }

                # rows_new.append(out)
                # _persist_row(out)
                # time.sleep(1)
finally:
    jf.close(); cf.close()

# --- Combine old successes + new results (successful or failed) ---
df_new = pd.DataFrame(rows_new)

# keep only successful new ones for the merged final DataFrame
# df_new_ok = df_new[df_new["error"].isna() | (df_new["error"] == "")]
df = pd.concat([previous_ok_df, df_new_ok], ignore_index=True)

print("Final combined successful rows:", df.shape)
df.head()

In [ ]:
df_final = previous_ok_df

In [ ]:
df_final = df

In [ ]:
previous_ok_df[previous_ok_df["image"]=='IMG_20250421_125020.jpg']


### Results

In [ ]:

def load_ground_truth(csv_path: Path):
    if not csv_path.exists():
        print("No ground_truth.csv found — skipping evaluation.")
        return None
    gt = pd.read_csv(csv_path)
    gt.columns = [c.strip().lower() for c in gt.columns]
    assert "filename" in gt.columns and "vegetation_percent" in gt.columns, "ground_truth.csv must have filename,vegetation_percent"
    return gt

def merge_preds_with_gt(df_preds: pd.DataFrame, gt: pd.DataFrame) -> pd.DataFrame:
    m = df_preds.copy()
    if "error" in m.columns:
        m = m[m["error"].isna()]
    for k in ["vegetation_percent", "confidence"]:
        if k in m.columns:
            m[k] = pd.to_numeric(m[k], errors="coerce")
    gt2 = gt.rename(columns={"filename": "image"})
    return m.merge(gt2, on="image", suffixes=("_pred", "_true"))

def metrics(y_true, y_pred):
    y_true = y_true.astype(float); y_pred = y_pred.astype(float)
    err = y_pred - y_true
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err**2)))
    bias = float(np.mean(err))
    ss_res = float(np.sum(err**2)); ss_tot = float(np.sum((y_true - np.mean(y_true))**2))
    r2 = 1 - ss_res/ss_tot if ss_tot > 0 else np.nan
    return {"MAE": mae, "RMSE": rmse, "Bias": bias, "R2": r2}

In [ ]:
# Make a clean copy; coerce numerics
res = df_final.copy()

# Normalize types
for col in ["vegetation_percent", "confidence", "reference", "latency_ms"]:
    if col in res.columns:
        res[col] = pd.to_numeric(res[col], errors="coerce")

# Keep only rows with a numeric prediction and reference
mask_ok = res["vegetation_percent"].notna() & res["reference"].notna()
res_ok = res[mask_ok].copy()

# Basic error columns
res_ok["err"] = res_ok["vegetation_percent"] - res_ok["reference"]
res_ok["abs_err"] = res_ok["err"].abs()
res_ok["sq_err"] = res_ok["err"]**2

print("Total rows:", len(res))
print("Valid rows (with pred & reference):", len(res_ok))
print("Unique images:", res_ok["image"].nunique())
print("Unique models:", res_ok["model"].nunique())
print("Unique prompts:", res_ok["prompt_id"].nunique())

In [ ]:
print("Unique models:", res_ok["model"].unique())

In [ ]:
# --- 1) Define bins and compute per-(model, prompt, bin) metrics ---
bins = np.arange(0, 101, 10)  # 0–20, 20–40, 40–60, 60–80, 80–100
labels = [f"{bins[i]}–{bins[i+1]}" for i in range(len(bins)-1)]
res_ok = res_ok.copy()
res_ok["ref_bin"] = pd.cut(res_ok["reference"], bins=bins, labels=labels, include_lowest=True, right=True)

by_mp_bin = (res_ok
    .groupby(["model", "prompt_id", "ref_bin"], dropna=False)
    .apply(lambda g: pd.Series(metrics(g["reference"], g["vegetation_percent"])))
    .reset_index())

# Optional: also keep counts used per cell (for QC / annotations)
counts_mp_bin = (res_ok
    .groupby(["model", "prompt_id", "ref_bin"], dropna=False)
    .size()
    .reset_index(name="n"))

by_mp_bin = by_mp_bin.merge(counts_mp_bin, on=["model","prompt_id","ref_bin"], how="left")

# --- 2) Prepare grid layout ---
models  = sorted(res_ok["model"].dropna().unique().tolist())
prompts = sorted(res_ok["prompt_id"].dropna().unique().tolist())

n_rows = len(models)
n_cols = len(prompts)

# Get a common max for MAE axis so scales are comparable across all cells
global_mae_max = by_mp_bin["MAE"].max()
if pd.isna(global_mae_max):
    global_mae_max = 1.0
ymax = max(global_mae_max * 1.15, 1.0)

# --- 3) Plot grid (rows=models, cols=prompts) ---
fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(4.5*n_cols, 3.8*n_rows),
    squeeze=False
)

for r, m in enumerate(models):
    for c, p in enumerate(prompts):
        ax = axes[r, c]
        sub = by_mp_bin[(by_mp_bin["model"] == m) & (by_mp_bin["prompt_id"] == p)]
        # Ensure bins appear in order with 0 for missing
        mae_vals = []
        ns = []
        for b in labels:
            row = sub[sub["ref_bin"] == b]
            mae_vals.append(row["MAE"].values[0] if len(row) else np.nan)
            ns.append(int(row["n"].values[0]) if len(row) else 0)

        x = np.arange(len(labels))
        ax.bar(x, mae_vals)
        ax.set_ylim(0, ymax)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=30, ha="right")
        if r == 0:
            ax.set_title(p)
        if c == 0:
            ax.set_ylabel(f"{m}\nMAE (%)")
        # Optional: annotate counts on top of bars
        for xi, (val, n) in enumerate(zip(mae_vals, ns)):
            if not np.isnan(val):
                ax.text(xi, val + ymax*0.02, f"n={n}", ha="center", va="bottom", fontsize=8)

fig.suptitle("MAE by Reference Cover Bin — per (Model × Prompt)", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- PARAMETERS ---
TOP_N = 4  # number of best models to average (by MAE)
THRESHOLD = 40  # flag outliers with |Pred−Ref| ≥ threshold %
PROMPTS_TO_SHOW = ["v1_point_hint", "v2_short","v3_detailed_ecology","v4_grid_overlay"]  # which prompts to display

# --- 1) Compute MAE per model to find top N ---
mae_by_model = (res_ok
    .groupby("model")
    .apply(lambda g: np.mean(np.abs(g["vegetation_percent"] - g["reference"])))
    .sort_values()
)
top_models = mae_by_model.head(TOP_N).index.tolist()
print(f"Top {TOP_N} models by MAE:\n", mae_by_model.head(TOP_N))

# --- 2) Average predictions of top N models per image/prompt ---
# keep only top models
subset = res_ok[res_ok["model"].isin(top_models)].copy()

# group by (image, prompt_id), average predictions/confidences
agg = (subset
    .groupby(["image", "prompt_id", "reference"], as_index=False)
    .agg({
        "vegetation_percent": "mean",
        "confidence": "mean",
        "model": lambda x: list(set(x))
    })
)
agg.rename(columns={"vegetation_percent":"pred_mean"}, inplace=True)

# compute abs error
agg["abs_error"] = np.abs(agg["pred_mean"] - agg["reference"])

# --- 3) Filter only selected prompts ---
if PROMPTS_TO_SHOW:
    agg = agg[agg["prompt_id"].isin(PROMPTS_TO_SHOW)]

# --- 4) Plot matrix (rows = prompts, single "average" model row) ---
prompts = sorted(agg["prompt_id"].dropna().unique())
n_rows = len(prompts)
fig, axes = plt.subplots(n_rows, 1, figsize=(6, 4.5*n_rows), squeeze=False)

out_v3 = []

for r, prompt_id in enumerate(prompts):
    ax = axes[r, 0]
    sub = agg[agg["prompt_id"] == prompt_id]
    outliers = sub[sub["abs_error"] >= THRESHOLD]
    if prompt_id == "v3_detailed_ecology":
        for _, row in outliers.iterrows():
            out_v3.append([row["image"],row["reference"],row["pred_mean"]])

    ax.scatter(sub["reference"], sub["pred_mean"], color="gray", s=20, alpha=0.5, label="All")
    ax.scatter(outliers["reference"], outliers["pred_mean"], color="red", s=35, alpha=0.9, label="Outliers")
    ax.plot([0,100],[0,100],"k--",lw=1)

    # optional labels
    for _, row in outliers.iterrows():
        ax.text(row["reference"]+1, row["pred_mean"]+1, row["image"], fontsize=7, alpha=0.8)

    n_all, n_out = len(sub), len(outliers)
    ax.text(3,95, f"n={n_all}, out={n_out}", fontsize=8,
            bbox=dict(facecolor="white", alpha=0.6, edgecolor="none"))

    ax.set_xlim(0,100)
    ax.set_ylim(0,100)
    ax.set_xlabel("Reference (%)")
    ax.set_ylabel("Predicted (avg of top models) (%)")
    ax.set_title(f"{prompt_id} (Avg of Top {TOP_N} models, |Pred−Ref| ≥ {THRESHOLD}%)")

plt.suptitle(f"Predicted vs Reference — Average of Top {TOP_N} Models", y=1.02, fontsize=14)
plt.tight_layout()
# plt.savefig(FIG_DIR / f"avg_top{TOP_N}_models_outlier_matrix_thr{THRESHOLD}.png", dpi=200)
plt.show()


In [ ]:
df_out_v3 = pd.DataFrame(out_v3, columns=['Image', 'Reference', 'Prediction'])
df_out_v3['Abs_error'] = np.abs(df_out_v3['Reference']-df_out_v3['Prediction'])
df_out_v3['Abs_error'] = np.abs(df_out_v3['Reference']-df_out_v3['Prediction'])
df_out_v3.sort_values(by='Abs_error',ascending=False)

In [ ]:
df_out_v3[df_out_v3["Image"]=='IMG_20250421_125020.jpg']
